## Step 1 - Data Collection and Data Storage

1.1) We'll use a public dataset to demonstrate the concepts. Let's use the Wine Quality Dataset from UCI Machine Learning Repository.

In [ ]:
import pandas as pd
import requests
from io import StringIO

def download_wine_data():
    # URLs for red and white wine datasets
    red_wine_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
    white_wine_url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-white.csv"

    # Download and read the datasets
    red_wine = pd.read_csv(StringIO(requests.get(red_wine_url).text), sep=';')
    white_wine = pd.read_csv(StringIO(requests.get(white_wine_url).text), sep=';')

    # Add a column to distinguish between red and white wines
    red_wine['wine_type'] = 'red'
    white_wine['wine_type'] = 'white'

    # Combine the datasets
    wine_data = pd.concat([red_wine, white_wine], ignore_index=True)

    return wine_data

# Download the data
wine_data = download_wine_data()

# Save to CSV
wine_data.to_csv('wine_quality_data.csv', index=False)

print("Data downloaded and saved to 'wine_quality_data.csv'")

Data downloaded and saved to 'wine_quality_data.csv'


In [ ]:
pd.read_csv('/content/drive/MyDrive/Integrated Project/wine_quality_data.csv')

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


1.2) We'll use SQLite as our database to store the structured data. It's lightweight and doesn't require a separate server setup.

In [ ]:
import sqlite3
import pandas as pd

def store_data_in_sqlite(csv_file, db_name, table_name):

    df = pd.read_csv(csv_file)

    # Connect to SQLite database (or create it if it doesn't exist)
    conn = sqlite3.connect(db_name)

    # Store the dataframe in SQLite
    df.to_sql(table_name, conn, if_exists='replace', index=False)

    print(f"Data stored in SQLite database '{db_name}' in table '{table_name}'")

    # Close the connection
    conn.close()

# Store the data
store_data_in_sqlite('/content/drive/MyDrive/Integrated Project/wine_quality_data.csv', 'wine_quality.db', 'wine_data')

Data stored in SQLite database 'wine_quality.db' in table 'wine_data'


In [ ]:


conn = sqlite3.connect('wine_quality.db')

query = "SELECT * FROM wine_data"
df = pd.read_sql_query(query, conn)

display(df)

conn.close()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,red
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,red
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,red
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,red
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,white
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,white
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,white
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,white


## Step 2 - Data Preprocessing and Exploration

Now that we have our data stored, let's preprocess it and perform some exploratory data analysis.

In [ ]:

import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

# Connect to the SQLite database and load the data
conn = sqlite3.connect('wine_quality.db')
df = pd.read_sql_query("SELECT * FROM wine_data", conn)
conn.close()

# Basic information about the dataset
print(df.info())
print("\nSample data:")
print(df.head())

# Check for missing values
print("\nMissing values:")
print(df.isnull().sum())

# Display summary statistics
print("\nSummary statistics:")
print(df.describe().round(2))

# Visualize the distribution of wine quality
plt.figure(figsize=(10, 6))
sns.countplot(x='quality', hue='wine_type', data=df)
plt.title('Distribution of Wine Quality by Type')
plt.savefig('wine_quality_distribution.png')
plt.close()

# Correlation heatmap
# Include only numeric features for correlation analysis
numeric_features = df.select_dtypes(include=['number'])

plt.figure(figsize=(12, 10))
sns.heatmap(numeric_features.corr(), annot=True, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Heatmap of Wine Features')
plt.savefig('correlation_heatmap.png')
plt.close()

# Preprocess the data
# Convert categorical variable to numeric
df['wine_type'] = df['wine_type'].map({'white': 0, 'red': 1})

# Separate features and target
X = df.drop('quality', axis=1)
y = df['quality']

# Scale the features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Save preprocessed data
preprocessed_df = pd.DataFrame(X_scaled, columns=X.columns)
preprocessed_df['quality'] = y
preprocessed_df.to_csv('preprocessed_wine_data.csv', index=False)

print("Preprocessing complete. Preprocessed data saved to 'preprocessed_wine_data.csv'")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6497 entries, 0 to 6496
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         6497 non-null   float64
 1   volatile acidity      6497 non-null   float64
 2   citric acid           6497 non-null   float64
 3   residual sugar        6497 non-null   float64
 4   chlorides             6497 non-null   float64
 5   free sulfur dioxide   6497 non-null   float64
 6   total sulfur dioxide  6497 non-null   float64
 7   density               6497 non-null   float64
 8   pH                    6497 non-null   float64
 9   sulphates             6497 non-null   float64
 10  alcohol               6497 non-null   float64
 11  quality               6497 non-null   int64  
 12  wine_type             6497 non-null   object 
dtypes: float64(11), int64(1), object(1)
memory usage: 660.0+ KB
None

Sample data:
   fixed acidity  volatile acidity  citric a

In [ ]:
pd.read_csv('preprocessed_wine_data.csv')

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,wine_type,quality
0,0.142473,2.188833,-2.192833,-0.744778,0.569958,-1.100140,-1.446359,1.034993,1.813090,0.193097,-0.915464,1.750190,5
1,0.451036,3.282235,-2.192833,-0.597640,1.197975,-0.311320,-0.862469,0.701486,-0.115073,0.999579,-0.580068,1.750190,5
2,0.451036,2.553300,-1.917553,-0.660699,1.026697,-0.874763,-1.092486,0.768188,0.258120,0.797958,-0.580068,1.750190,5
3,3.073817,-0.362438,1.661085,-0.744778,0.541412,-0.762074,-0.986324,1.101694,-0.363868,0.327510,-0.580068,1.750190,6
4,0.142473,2.188833,-2.192833,-0.744778,0.569958,-1.100140,-1.446359,1.034993,1.813090,0.193097,-0.915464,1.750190,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,-0.783214,-0.787650,-0.197054,-0.807837,-0.486252,-0.367664,-0.420128,-1.186161,0.320319,-0.210144,0.593818,-0.571367,6
6493,-0.474652,-0.119460,0.284686,0.537425,-0.257883,1.491697,0.924588,0.067824,-0.426067,-0.478971,-0.747766,-0.571367,5
6494,-0.551792,-0.605417,-0.885253,-0.891916,-0.429160,-0.029599,-0.083949,-0.719251,-1.421248,-0.478971,-0.915464,-0.571367,6
6495,-1.323198,-0.301694,-0.128234,-0.912936,-0.971538,-0.593041,-0.101642,-2.003251,0.755710,-1.016626,1.935402,-0.571367,7


In [ ]:
df

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,wine_type
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,1
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.99680,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.99700,3.26,0.65,9.8,5,1
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.99800,3.16,0.58,9.8,6,1
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.99780,3.51,0.56,9.4,5,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6492,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6,0
6493,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5,0
6494,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6,0
6495,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7,0


## Step 3 - Feature Engineering

Feature engineering is crucial for improving model performance. We'll create new features and select the most important ones.

1. Loads the preprocessed data

2. Creates interaction features with alcohol content

3. Generates polynomial features

4. Combines all features

5. Selects the top K most important features

6. Saves the engineered dataset

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.impute import SimpleImputer

# Load the preprocessed data
df = pd.read_csv('preprocessed_wine_data.csv')

# Separate features and target
X = df.drop('quality', axis=1)
y = df['quality']

# Creates interaction features with alcohol content
def create_interaction_features(X):
    return X.apply(lambda x: x * X['alcohol'], axis=1).add_suffix('_alcohol_interaction')

# Generate interaction features
interaction_features = create_interaction_features(X)

# Create polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)
X_poly = poly.fit_transform(X)
poly_feature_names = poly.get_feature_names_out(X.columns)
poly_features = pd.DataFrame(X_poly, columns=poly_feature_names)

# Combine original, interaction, and polynomial features
X_all_features = pd.concat([X, interaction_features, poly_features], axis=1)

# Drop columns with all missing or zero values
X_all_features_cleaned = X_all_features.replace([np.inf, -np.inf], np.nan).dropna(axis=1, how='all')

# Impute missing values using the mean
imputer = SimpleImputer(strategy='mean')
X_all_features_imputed = imputer.fit_transform(X_all_features_cleaned)

# If the number of columns doesn't change after imputation, apply original feature names
if X_all_features_imputed.shape[1] == X_all_features_cleaned.shape[1]:
    X_all_features_imputed = pd.DataFrame(X_all_features_imputed, columns=X_all_features_cleaned.columns)
else:
    X_all_features_imputed = pd.DataFrame(X_all_features_imputed)

# Keep track of the feature names
original_features = X_all_features_cleaned.columns

# Select top K features using f_regression
k = 10
selector = SelectKBest(score_func=f_regression, k=k)
X_selected = selector.fit_transform(X_all_features_imputed, y)

# Get selected feature indices and names
selected_indices = selector.get_support()
selected_features = [original_features[i] for i in range(len(original_features)) if selected_indices[i]]

# Create final dataframe with selected features
X_final = pd.DataFrame(X_selected, columns=selected_features)
X_final['quality'] = y

# Save the engineered features to a CSV file
X_final.to_csv('engineered_wine_data.csv', index=False)

# Print results
print(f"Feature engineering complete. Selected {k} features:")
print(selected_features)
print("\nEngineered data saved to 'engineered_wine_data.csv'")


Feature engineering complete. Selected 10 features:
['volatile acidity', 'chlorides', 'density', 'alcohol', 'volatile acidity', 'chlorides', 'density', 'alcohol', 'density alcohol', 'alcohol^2']

Engineered data saved to 'engineered_wine_data.csv'


In [ ]:
pd.read_csv('engineered_wine_data.csv')

,volatile acidity,chlorides,density,alcohol,volatile acidity.1,chlorides.1,density.1,alcohol.1,density alcohol,alcohol^2,quality
0,2.188833,0.569958,1.034993,-0.915464,2.188833,0.569958,1.034993,-0.915464,-0.947499,0.838075,5
1,3.282235,1.197975,0.701486,-0.580068,3.282235,1.197975,0.701486,-0.580068,-0.406910,0.336479,5
2,2.553300,1.026697,0.768188,-0.580068,2.553300,1.026697,0.768188,-0.580068,-0.445601,0.336479,5
3,-0.362438,0.541412,1.101694,-0.580068,-0.362438,0.541412,1.101694,-0.580068,-0.639058,0.336479,6
4,2.188833,0.569958,1.034993,-0.915464,2.188833,0.569958,1.034993,-0.915464,-0.947499,0.838075,5
...,...,...,...,...,...,...,...,...,...,...,...
6492,-0.787650,-0.486252,-1.186161,0.593818,-0.787650,-0.486252,-1.186161,0.593818,-0.704363,0.352620,6
6493,-0.119460,-0.257883,0.067824,-0.747766,-0.119460,-0.257883,0.067824,-0.747766,-0.050716,0.559154,5
6494,-0.605417,-0.429160,-0.719251,-0.915464,-0.605417,-0.429160,-0.719251,-0.915464,0.658449,0.838075,6
6495,-0.301694,-0.971538,-2.003251,1.935402,-0.301694,-0.971538,-2.003251,1.935402,-3.877097,3.745781,7


## Step 4 - Model Development

Now that we have our engineered features, let's develop and compare multiple models.

The following has been performed in it:


1. Loads the engineered data

2. Splits the data into training and test sets

3. Defines multiple models (Linear Regression, Random Forest, SVR, Neural Network)

4. Trains each model and evaluates its performance

5. Performs cross-validation

6. Saves each trained model

7. Identifies the best model based on cross-validation results

8. Makes final predictions using the best model

9. Saves the final predictions

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# Load the engineered data
df = pd.read_csv('engineered_wine_data.csv')

# Separate features and target
X = df.drop('quality', axis=1)
y = df['quality']

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define models
models = {
    'Linear Regression': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
    'SVR': SVR(kernel='rbf'),
    'Neural Network': MLPRegressor(hidden_layer_sizes=(100, 50), max_iter=500)
}

# Train and evaluate models
results = {}

for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)

    # Make predictions
    y_pred = model.predict(X_test)

    # Calculate metrics
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    # Perform cross-validation
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error')

    # Store results
    results[name] = {
        'MSE': mse,
        'R2': r2,
        'CV_MSE': -cv_scores.mean()
    }

    # Save the model
    joblib.dump(model, f'{name.replace(" ", "_").lower()}_model.joblib')

# Print results
print("Model Evaluation Results:")
for name, metrics in results.items():
    print(f"\n{name}:")
    print(f"  MSE: {metrics['MSE']:.4f}")
    print(f"  R2: {metrics['R2']:.4f}")
    print(f"  Cross-validation MSE: {metrics['CV_MSE']:.4f}")

# Identify the best model
best_model = min(results, key=lambda x: results[x]['CV_MSE'])
print(f"\nBest model based on cross-validation: {best_model}")

# Final prediction using the best model
best_model = joblib.load(f'{best_model.replace(" ", "_").lower()}_model.joblib')
final_predictions = best_model.predict(X_test)

# Save final predictions
pd.DataFrame({'Actual': y_test, 'Predicted': final_predictions}).to_csv('final_predictions.csv', index=False)
print("\nFinal predictions saved to 'final_predictions.csv'")

Model Evaluation Results:

Linear Regression:
  MSE: 0.5489
  R2: 0.2567
  Cross-validation MSE: 0.5709

Random Forest:
  MSE: 0.4358
  R2: 0.4099
  Cross-validation MSE: 0.6288

SVR:
  MSE: 0.5490
  R2: 0.2566
  Cross-validation MSE: 0.5843

Neural Network:
  MSE: 0.5377
  R2: 0.2719
  Cross-validation MSE: 0.5793

Best model based on cross-validation: Linear Regression

Final predictions saved to 'final_predictions.csv'


In [ ]:
pd.read_csv('final_predictions.csv')

,Actual,Predicted
0,8,6.685919
1,5,5.127758
2,7,6.196233
3,6,5.567404
4,6,5.451469
...,...,...
1295,5,4.952689
1296,5,5.364427
1297,7,6.700255
1298,6,5.476053


## Step 5 - Big Data Processing

For big data processing, we'll use PySpark to demonstrate how to handle large-scale data. We'll create a simple example of how to use Spark for data processing and model training.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize Spark session
spark = SparkSession.builder.appName("WineQualityPrediction").getOrCreate()

# Load data
df = spark.read.csv('engineered_wine_data.csv', header=True, inferSchema=True)

# Prepare features
feature_columns = df.columns
feature_columns.remove('quality')
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
data = assembler.transform(df)

# Split the data
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

# Train a Random Forest model
rf = RandomForestRegressor(featuresCol="features", labelCol="quality")
model = rf.fit(train_data)

# Make predictions
predictions = model.transform(test_data)

# Evaluate the model
evaluator = RegressionEvaluator(labelCol="quality", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"Root Mean Squared Error (RMSE) on test data = {rmse}")


# Save the model
model.write().overwrite().save("spark_rf_model")

# Stop the Spark session
spark.stop()

Root Mean Squared Error (RMSE) on test data = 0.7198050861739608


## Step 6 - Real Time Analytics

For real-time analytics, we'll create a simple Flask API that can make predictions using our trained model. This simulates how you might deploy your model for real-time use.

In [ ]:
from flask import Flask, request, jsonify
import joblib
import pandas as pd

app = Flask(__name__)

# Load the trained model
model = joblib.load('random_forest_model.joblib')  # Adjust the filename if needed

@app.route('/predict', methods=['POST'])
def predict():
    # Get the data from the POST request
    data = request.get_json(force=True)

    # Convert data to DataFrame
    df = pd.DataFrame(data, index=[0])

    # Make prediction
    prediction = model.predict(df)

    # Return the prediction
    return jsonify({'prediction': prediction[0]})

if __name__ == '__main__':
    app.run(port=5000, debug=True)

 * Serving Flask app '__main__'
 * Debug mode: on


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:5000
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug: * Restarting with stat


## Step 9 - Streamlit Visualization